In [2]:
from pyrosetta import init, pose_from_pdb
from pyrosetta.rosetta.core.scoring import hbonds
from pyrosetta.rosetta.core.scoring import ScoreFunction
from pyrosetta.rosetta.protocols.analysis import InterfaceAnalyzerMover, InterfaceData
import os
import json
from pyrosetta import *
from pyrosetta.teaching import *
# Initialize PyRosetta
init()
def get_chain_ids_from_pdb(pdb_file):
    chain_ids = []
    with open(pdb_file, "r") as f:
        for line in f:
            if line.startswith("ATOM") or line.startswith("HETATM"):
                chain_id = line[21]  # 체인 ID는 22번째 컬럼 (0-index 기준 21)
                if chain_id not in chain_ids:
                    chain_ids.append(chain_id)
                if len(chain_ids) == 2:  # 두 개만 찾으면 종료
                    break
    return chain_ids

def get_residues_by_chain(pose, chain_id):
    residues = []
    for res in range(1, pose.total_residue() + 1):
        if pose.pdb_info().chain(res) == chain_id:
            residues.append(res)
    return residues

# Directories
pdb_dir = '/home/psh/protein-frame-flow/inference_outputs/fm_tri_crop_450/2025-03-27_22-56-13/epoch=107-step=122256/run_2025-04-01_14-47-31/8slb_H_L_A'


# Process each PDB file
for sample in os.listdir(pdb_dir):

    pdb_file = os.path.join(pdb_dir, sample, 'sample_1.pdb')
    ag = os.path.basename(pdb_dir).split('_')[-1]

    chains = get_chain_ids_from_pdb(pdb_file)
    # Load PDB file into PyRosetta pose
    pose = pose_from_pdb(pdb_file)

    # Setup and apply InterfaceAnalyzerMover
    interface_analyzer = InterfaceAnalyzerMover()
    interface_data = InterfaceData()
    interface_analyzer.set_interface(f"{chains[0]}{chains[1]}_{ag}")
    interface_analyzer.set_calc_hbond_sasaE(True)
    interface_analyzer.set_compute_interface_energy(True)
    interface_analyzer.set_compute_interface_sc(True)
    interface_analyzer.set_compute_packstat(True)
    interface_analyzer.apply(pose)
    interface_data = interface_analyzer.get_all_data()

    # Extract interface energy
    interface_energy = interface_analyzer.get_interface_dG()
    sasa = interface_analyzer.get_interface_delta_sasa()
    packstat = interface_analyzer.get_interface_packstat()

    output_data = {
        'interface_dG': interface_energy,
        'hbonds_num': interface_data.interface_hbonds,
        'hbonds_total_E': interface_data.total_hb_E,
        'hbond_E_fraction': interface_data.hbond_E_fraction,
        'sc_value': interface_data.sc_value, # shape complementarity score (높을수록 interaction하는 두 분자의 표면이 잘 맞아떨어진다는 의미), 0~1
        'packstat_score': packstat,
        'sasa': sasa
    }
    # Save to JSON file
    json_file = os.path.join(pdb_dir, sample, 'inter_energy.json')
    with open(json_file, 'w') as f:
        json.dump(output_data, f, indent=4)

    print(f"Saved interface energy for {sample} to {json_file}")


┌──────────────────────────────────────────────────────────────────────────────┐
│                                 PyRosetta-4                                  │
│              Created in JHU by Sergey Lyskov and PyRosetta Team              │
│              (C) Copyright Rosetta Commons Member Institutions               │
│                                                                              │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRE PURCHASE OF A LICENSE │
│         See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└──────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2024 [Rosetta PyRosetta4.Release.python312.linux 2024.19+release.a34b73c40fe9c61558d566d6a63f803cfb15a4fc 2024-05-02T16:22:03] retrieved from: http://www.pyrosetta.org
core.init: Checking for fconfig files in pwd and ./rosetta/flags
core.init: Rosetta version: PyRosetta4.Release.python312.linux r381 2024.19+release.a34b73c a34b73c40fe9c61

core.chemical.GlobalResidueTypeSet: Finished initializing fa_standard residue type set.  Created 985 residue types
core.chemical.GlobalResidueTypeSet: Total time to initialize 0.737154 seconds.
core.import_pose.import_pose: File '/home/psh/protein-frame-flow/inference_outputs/fm_tri_crop_450/2025-03-27_22-56-13/epoch=107-step=122256/run_2025-04-01_14-47-31/8slb_H_L_A/sample_0/sample_1.pdb' automatically determined to be of type PDB
core.conformation.Conformation: [ WARNING ] missing heavyatom:  OXT on residue ASN:CtermProteinFull 130
core.conformation.Conformation: [ WARNING ] missing heavyatom:  OXT on residue LYS:CtermProteinFull 237
core.conformation.Conformation: [ WARNING ] missing heavyatom:  OXT on residue GLU:CtermProteinFull 485
core.conformation.Conformation: Found disulfide between residues 22 96
protocols.analysis.InterfaceAnalyzerMover: Using explicit constructor
protocols.analysis.InterfaceAnalyzerMover: Using interface constructor
protocols.evaluation.ChiWellRmsdEvaluato